# Exercise 06 — GPU-Accelerated Spike-Train Cross-Correlation

## Background

**Cross-correlation** between two spike trains measures the temporal relationship between their firing:
$$C_{ij}(\tau) = \sum_t s_i(t) \cdot s_j(t + \tau)$$

where $\tau$ is the lag (in ms). The cross-correlogram shows whether neuron j tends to fire $\tau$ ms after neuron i.

**Two approaches:**
1. **Direct:** For each spike pair, compute lag and bin. O(n_i × n_j) per pair.
2. **FFT-based:** C = IFFT(FFT(s_i) × conj(FFT(s_j))). O(N log N) per pair.

The FFT approach is faster for long recordings. Your task is to implement it using cuFFT.

## Task

Complete the GPU pipeline to compute all N×N pairwise cross-correlograms for N=50 neurons.

In [ ]:
!nvidia-smi

In [ ]:
%%writefile xcorr_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cufft.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)
#define CUFFT_CHECK(call) do { cufftResult e=(call); \
    if(e!=CUFFT_SUCCESS){fprintf(stderr,"cuFFT %d at %s:%d\n",e,__FILE__,__LINE__);exit(1);}} while(0)

// ─────────────────────────────────────────────────────────────────────────────
// PART 1: Element-wise complex multiply with conjugate
// For each frequency bin k and each pair (i, j):
//   out[pair * n_freq + k] = X[i * n_freq + k] * conj(X[j * n_freq + k])
// where pair = i * N + j
//
// This computes the cross-power spectrum for all pairs simultaneously.
// ─────────────────────────────────────────────────────────────────────────────
__global__ void cross_power_all_pairs(
    const cufftComplex* X,  // [N_neurons × n_freq] FFTs of all spike trains
    cufftComplex* out,      // [N² × n_freq] output cross-power spectra
    int N,                  // number of neurons
    int n_freq              // number of frequency bins = T_bins/2 + 1
) {
    // Thread (i, j, k): neuron pair (i,j) at frequency bin k
    int k    = blockIdx.x * blockDim.x + threadIdx.x;
    int pair = blockIdx.y;  // pair index 0..N²-1
    if (k >= n_freq || pair >= N*N) return;

    int i = pair / N;  // pre-synaptic neuron
    int j = pair % N;  // post-synaptic neuron

    // ??? Load X_i[k] and X_j[k]
    cufftComplex xi = ???;
    cufftComplex xj = ???;

    // ??? Compute xi * conj(xj):
    //   real part = xi.x*xj.x + xi.y*xj.y
    //   imag part = xi.y*xj.x - xi.x*xj.y
    cufftComplex result;
    result.x = ???;
    result.y = ???;

    // ??? Store result
    out[???] = result;
}

// ─────────────────────────────────────────────────────────────────────────────
// PART 2: Normalize cross-correlogram
// After IFFT, the cross-correlogram has values in the range [0, N_spikes].
// Normalize by T_bins (length) to get the expected correlation per bin.
// Also, shift the result so that lag=0 is at the center.
// ─────────────────────────────────────────────────────────────────────────────
__global__ void normalize_xcorr(
    float* xcorr,  // [N² × T_bins] — real part of IFFT output
    int N, int T_bins, float scale
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= N * N * T_bins) return;

    // ??? Multiply by scale (= 1.0f / T_bins)
    xcorr[idx] ???;
}

int main(int argc, char** argv)
{
    int N_neurons = (argc>1) ? atoi(argv[1]) : 50;
    int T_bins    = (argc>2) ? atoi(argv[2]) : 1000;  // 1000 ms at 1 ms bins
    float rate    = 0.05f;  // 5% firing probability per bin
    int n_freq    = T_bins / 2 + 1;
    int n_pairs   = N_neurons * N_neurons;

    printf("Cross-correlation: %d neurons × %d ms\n", N_neurons, T_bins);
    printf("Pairs: %d, freq bins: %d\n", n_pairs, n_freq);

    // Generate synthetic spike trains
    float* h_spikes = (float*)calloc((size_t)N_neurons * T_bins, sizeof(float));
    srand(42);
    for (int i = 0; i < N_neurons * T_bins; i++)
        if ((float)rand()/RAND_MAX < rate) h_spikes[i] = 1.0f;

    // GPU arrays
    float       *d_spikes, *d_xcorr;
    cufftComplex *d_fft_spikes, *d_cross_power, *d_xcorr_complex;

    CUDA_CHECK(cudaMalloc(&d_spikes,       (size_t)N_neurons * T_bins   * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_fft_spikes,   (size_t)N_neurons * n_freq   * sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_cross_power,  (size_t)n_pairs   * n_freq   * sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_xcorr_complex,(size_t)n_pairs   * T_bins   * sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_xcorr,        (size_t)n_pairs   * T_bins   * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(d_spikes, h_spikes,
                          (size_t)N_neurons * T_bins * sizeof(float),
                          cudaMemcpyHostToDevice));

    // ─── Step 1: FFT all spike trains ────────────────────────────────────────
    // ??? Create a batch FFT plan: N_neurons transforms of length T_bins
    cufftHandle plan_fwd;
    CUFFT_CHECK(cufftPlan1d(&plan_fwd, T_bins, CUFFT_R2C, ???));
    CUFFT_CHECK(cufftExecR2C(plan_fwd, d_spikes, d_fft_spikes));

    // ─── Step 2: Compute all cross-power spectra ──────────────────────────────
    int thr = 256;
    int blk_freq = (n_freq + thr - 1) / thr;

    // ??? Launch cross_power_all_pairs
    // Hint: use a 2D grid — blockIdx.x for freq bins, blockIdx.y for pairs
    dim3 grid(???, n_pairs);
    cross_power_all_pairs<<<grid, thr>>>(d_fft_spikes, d_cross_power,
                                         N_neurons, n_freq);

    // ─── Step 3: Inverse FFT to get cross-correlograms ────────────────────────
    // ??? Create inverse FFT plan: n_pairs transforms of length T_bins
    cufftHandle plan_inv;
    CUFFT_CHECK(cufftPlan1d(&plan_inv, T_bins, CUFFT_C2C, ???));
    CUFFT_CHECK(cufftExecC2C(plan_inv, d_cross_power, d_xcorr_complex, CUFFT_INVERSE));

    // ─── Step 4: Extract real part and normalize ─────────────────────────────
    // The real part of IFFT(X_i * conj(X_j)) is the cross-correlogram
    // (Imaginary part should be ~0 for real signals)

    // ??? Call normalize_xcorr to scale by 1/T_bins
    // For simplicity: treat d_xcorr_complex's real components as floats
    // (They are interleaved: [re0, im0, re1, im1, ...])
    // Easier: copy and extract in Python

    // Save output for Python visualization
    cufftComplex* h_xcorr_c = (cufftComplex*)malloc(
        (size_t)n_pairs * T_bins * sizeof(cufftComplex));
    CUDA_CHECK(cudaMemcpy(h_xcorr_c, d_xcorr_complex,
                          (size_t)n_pairs * T_bins * sizeof(cufftComplex),
                          cudaMemcpyDeviceToHost));

    // Save auto-correlogram (pair i=0,j=0) and cross-correlogram (i=0,j=1)
    FILE* f = fopen("xcorr_results.txt", "w");
    fprintf(f, "# auto-corr (0,0) then cross-corr (0,1)\n");
    int pair_00 = 0 * N_neurons + 0;
    int pair_01 = 0 * N_neurons + 1;
    for (int t = 0; t < T_bins; t++) {
        float ac = h_xcorr_c[pair_00 * T_bins + t].x / T_bins;
        float cc = h_xcorr_c[pair_01 * T_bins + t].x / T_bins;
        fprintf(f, "%d %.6f %.6f\n", t - T_bins/2, ac, cc);
    }
    fclose(f);
    printf("Results saved to xcorr_results.txt\n");

    CUFFT_CHECK(cufftDestroy(plan_fwd));
    CUFFT_CHECK(cufftDestroy(plan_inv));
    cudaFree(d_spikes); cudaFree(d_fft_spikes);
    cudaFree(d_cross_power); cudaFree(d_xcorr_complex); cudaFree(d_xcorr);
    free(h_spikes); free(h_xcorr_c);
    return 0;
}

In [ ]:
!nvcc -O2 -o xcorr_gpu xcorr_gpu.cu -lcufft -lm && ./xcorr_gpu 50 2000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('xcorr_results.txt')
lags, auto_corr, cross_corr = data[:, 0], data[:, 1], data[:, 2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Auto-correlogram (should peak at lag=0)
ax1.bar(lags, auto_corr, width=1, color='steelblue', alpha=0.7)
ax1.set_xlim(-100, 100)
ax1.set_xlabel('Lag (ms)', fontsize=12)
ax1.set_ylabel('Correlation', fontsize=12)
ax1.set_title('Auto-Correlogram (neuron 0)', fontsize=12)
ax1.axvline(0, color='r', linestyle='--', lw=1)
ax1.grid(True, alpha=0.3)

# Cross-correlogram (should be ~flat for independent neurons)
ax2.bar(lags, cross_corr, width=1, color='coral', alpha=0.7)
ax2.set_xlim(-100, 100)
ax2.set_xlabel('Lag (ms)', fontsize=12)
ax2.set_ylabel('Correlation', fontsize=12)
ax2.set_title('Cross-Correlogram (neuron 0 → 1)', fontsize=12)
ax2.axvline(0, color='b', linestyle='--', lw=1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Reflection Questions

1. **Memory:** How much GPU memory does the `d_cross_power` buffer require for N=50 neurons, T=2000 bins? What is the maximum N you could use on a T4 GPU with 16 GB?

2. **Complexity:** Compare the complexity of the FFT approach (O(N² × T log T)) vs the direct approach (O(N² × n_spikes × τ_max/dt)). At what firing rate and recording length does FFT win?

3. **Normalization:** The raw IFFT output has values in [0, T]. How would you normalize to get a correlation coefficient in [-1, 1]? What additional quantities do you need?

4. **Extension:** How would you modify the code to compute only the lower triangle of the N×N correlation matrix (since C_ij(τ) = C_ji(-τ))?

## Challenge

Add a shared oscillation to some neurons to create a non-flat cross-correlogram. Specifically:
- Neurons 0–24: fire randomly at 5%, PLUS fire whenever a global 10 Hz rhythm fires
- Neurons 25–49: fire randomly at 5%

Show that the cross-correlogram between neurons from the first group has an oscillatory peak at ±100 ms (10 Hz period), while cross-correlograms between groups are flat.